# 🧩 Notebook 3: Real-World Sidecar Patterns

In production, sidecars usually do more than just auth. This notebook walks through three patterns you'll see in the wild, plus the trade-offs of using sidecars at all.

1. **Proxy sidecar** with retries + timeouts (the Envoy / linkerd-proxy role)
2. **Log-forwarder sidecar** (the Fluent Bit / Vector role)
3. **Config-refresher sidecar** (the Vault Agent / consul-template role)

Each pattern uses only the Python standard library, so you can run everything from the notebook.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

### Shared helpers (idempotent server start, like Notebook 2)

In [ ]:
import threading, time, json, random, os, tempfile, urllib.request, urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

_servers = {}
def start_server(name, port, handler_cls):
    if name in _servers:
        return _servers[name]
    srv = ThreadingHTTPServer(('127.0.0.1', port), handler_cls)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    _servers[name] = srv
    time.sleep(0.15)
    return srv

## 1) Proxy sidecar with retries + timeouts

A flaky upstream is a fact of life in distributed systems (network blips, GC pauses, cold starts). A proxy sidecar can retry transient failures **without the app doing anything**.

**Bad**: app retries in every language, each team picks different policies.
**Best**: the sidecar enforces one retry + timeout policy for the whole fleet. You can tune it centrally and get a consistent view in metrics.

### Flaky upstream app — fails ~40% of the time

In [ ]:
class FlakyApp(BaseHTTPRequestHandler):
    def do_GET(self):
        if random.random() < 0.4:  # 40% transient failure
            self.send_response(503); self.end_headers(); self.wfile.write(b'busy'); return
        body = json.dumps({'ok': True, 'path': self.path}).encode()
        self.send_response(200); self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body)
    def log_message(self, *a, **kw): pass

start_server('flaky-app', 9101, FlakyApp)
print('flaky app on :9101')

### Retry-proxy sidecar — up to 3 attempts, 500 ms budget each

In [ ]:
RETRYABLE_STATUS = {408, 425, 429, 500, 502, 503, 504}

class RetryProxy(BaseHTTPRequestHandler):
    UPSTREAM      = 'http://127.0.0.1:9101'
    MAX_ATTEMPTS  = 3
    TIMEOUT_S     = 0.5     # per-attempt timeout
    BUDGET_S      = 1.5     # total time budget across ALL attempts

    def do_GET(self):
        deadline = time.time() + self.BUDGET_S
        last = None
        for attempt in range(1, self.MAX_ATTEMPTS + 1):
            remaining = deadline - time.time()
            if remaining <= 0:
                last = f'budget of {self.BUDGET_S}s exhausted'
                break
            try:
                with urllib.request.urlopen(f'{self.UPSTREAM}{self.path}',
                                            timeout=min(self.TIMEOUT_S, remaining)) as r:
                    body = r.read()
                    print(f'[proxy] OK on attempt {attempt}')
                    self.send_response(200)
                    self.send_header('Content-Length', str(len(body)))
                    self.end_headers(); self.wfile.write(body); return
            except urllib.error.HTTPError as e:
                # ⚠️ The bug this guards against: retrying a 400/404/422 wastes the
                # budget on an answer that will never change, and turns one client's
                # bad request into 3x load on a healthy upstream.
                if e.code not in RETRYABLE_STATUS:
                    print(f'[proxy] {e.code} is terminal — passing it straight through')
                    body = e.read()
                    self.send_response(e.code)
                    self.send_header('Content-Length', str(len(body)))
                    self.end_headers(); self.wfile.write(body); return
                last = f'HTTP {e.code}'
            except Exception as e:
                last = str(e)                      # connection error / timeout
            if attempt == self.MAX_ATTEMPTS:
                break                              # no sleep after the last attempt
            backoff = min(0.05 * attempt + random.random() * 0.05,
                          max(0.0, deadline - time.time()))
            time.sleep(backoff)
        print(f'[proxy] gave up after {attempt} attempt(s): {last}')
        self.send_response(502); self.end_headers(); self.wfile.write(b'upstream failed')

    def log_message(self, *a, **kw): pass

start_server('retry-proxy', 9100, RetryProxy)
print('retry proxy on :9100')

In [ ]:
def hit(url, n=20):
    ok = 0
    for _ in range(n):
        try:
            urllib.request.urlopen(url, timeout=2).read()
            ok += 1
        except urllib.error.HTTPError:
            pass
    return ok

random.seed(1)
N = 20
direct = hit('http://127.0.0.1:9101/ping', N)   # straight at the flaky app
print()
proxied = hit('http://127.0.0.1:9100/ping', N)  # through the retry sidecar

print(f'\nno sidecar  : {direct}/{N} succeeded  (~60% expected: the app fails 40%)')
print(f'with sidecar: {proxied}/{N} succeeded  '
      f'(~94% expected: 1 - 0.4^3, three independent attempts)')
print()
print('The app did not change. One sidecar, one policy, and the whole fleet —')
print('in every language — inherits it.')

### Proof that terminal errors are not retried

The upstream below always answers `400 Bad Request`. A proxy that retried blindly would
send three requests and burn its whole budget on an answer that cannot change.

In [ ]:
class AlwaysBadRequest(BaseHTTPRequestHandler):
    hits = 0
    def do_GET(self):
        AlwaysBadRequest.hits += 1
        self.send_response(400); self.end_headers(); self.wfile.write(b'bad request')
    def log_message(self, *a, **kw): pass

start_server('bad-request-app', 9102, AlwaysBadRequest)

class TerminalProxy(RetryProxy):
    UPSTREAM = 'http://127.0.0.1:9102'
start_server('terminal-proxy', 9103, TerminalProxy)

AlwaysBadRequest.hits = 0
t0 = time.time()
try:
    urllib.request.urlopen('http://127.0.0.1:9103/ping', timeout=3).read()
except urllib.error.HTTPError as e:
    print(f'client saw: HTTP {e.code}')
print(f'upstream was called {AlwaysBadRequest.hits} time(s) in '
      f'{(time.time()-t0)*1000:.0f} ms  ← not 3, and the 400 was passed through intact')

### ⚠️ Only retry **idempotent** requests

Retries are safe when doing the same call twice has the same effect as doing it once — e.g. `GET /user/42`, `PUT /user/42 {...}`. They are **dangerous** for operations like `POST /payments` or `POST /send-email`, where a retry may charge the card twice.

In production sidecars (Envoy, linkerd-proxy) you configure *which* methods / status codes are retryable, often combined with an **idempotency key** header so the upstream can deduplicate. Rules of thumb:
- Retry only on `GET`, `HEAD`, `PUT`, `DELETE`, or `POST` with an idempotency key.
- Retry only on transient errors: connection failures, `502`, `503`, `504`, timeouts.
- Cap total retries **and** total time budget — otherwise slow upstreams cause retry storms.
- Add **jitter** to backoff (we do) so a fleet doesn't synchronise its retries into a thundering herd.

## 2) Log-forwarder sidecar

A common sidecar job: the app writes logs to a file (or stdout), and a sidecar ships them somewhere — e.g. Elasticsearch, Loki, S3. The app doesn't know or care about the log backend.

**Bad**: app directly calls the log backend's SDK → tight coupling, language-specific libraries, hard to rotate credentials.
**Best**: app writes structured lines to a shared file; sidecar tails the file and forwards them.

In [ ]:
# The app and the sidecar share a file (in a real pod: a shared `emptyDir` volume).
log_path = os.path.join(tempfile.gettempdir(), 'sidecar_demo.log')
open(log_path, 'w').close()  # truncate on (re)run

# --- the app just appends JSON lines ---
def app_emit(event):
    with open(log_path, 'a') as f:
        f.write(json.dumps(event) + '\n')

# --- the sidecar tails the file and "ships" each line ---
shipped = []  # stands in for Elasticsearch / Loki
def log_shipper():
    with open(log_path, 'r') as f:
        f.seek(0, os.SEEK_END)  # start at end, like `tail -F`
        while True:
            line = f.readline()
            if not line:
                time.sleep(0.05)
                continue
            shipped.append(json.loads(line))

threading.Thread(target=log_shipper, daemon=True).start()

# app writes 5 events
for i in range(5):
    app_emit({'lvl': 'info', 'i': i, 'msg': 'request handled'})

time.sleep(0.3)  # let the sidecar catch up
print('shipped events:', shipped)

## 3) Config-refresh sidecar

Secrets and configs rotate. You don't want to redeploy the app every time a token changes. A sidecar can watch for new values (e.g. from Vault or a config server) and write them to a shared file the app re-reads.

**Bad**: the app embeds a secret at build time.
**Best**: the sidecar owns secret rotation; the app only reads the current value from disk.

In [ ]:
cfg_path = os.path.join(tempfile.gettempdir(), 'sidecar_token')
def atomic_write(path, content):
    # write-to-tmp + rename = readers never see a half-written file
    tmp = path + '.tmp'
    with open(tmp, 'w') as f: f.write(content)
    os.replace(tmp, path)

atomic_write(cfg_path, 'token-v1')

# --- sidecar: rotate the token every 150 ms (simulating Vault lease renewal) ---
stop_rotator = threading.Event()
def rotator():
    n = 1
    while not stop_rotator.is_set():
        n += 1
        atomic_write(cfg_path, f'token-v{n}')
        time.sleep(0.15)
threading.Thread(target=rotator, daemon=True).start()

# --- app: read the token freshly each time it needs one ---
def app_current_token():
    with open(cfg_path) as f: return f.read().strip()

seen = []
for _ in range(5):
    seen.append(app_current_token())
    time.sleep(0.18)

stop_rotator.set()
print('tokens the app observed:', seen)

## 🌍 Real-world sidecars you'll meet

| Sidecar | Role | Shipped with |
|---|---|---|
| **Envoy** | L7 proxy: mTLS, retries, traffic split, tracing | Istio, Consul Connect, AWS App Mesh |
| **linkerd-proxy** | Rust micro-proxy focused on simplicity | Linkerd |
| **Fluent Bit / Vector** | Tail container logs, ship to a backend | Most Kubernetes log stacks |
| **Vault Agent** | Fetch and rotate secrets into a shared file | HashiCorp Vault |
| **OpenTelemetry Collector** | Receive/process/export traces & metrics | OpenTelemetry |
| **cloud-sql-proxy / AlloyDB auth proxy** | Secure DB auth without embedding credentials | GCP |

## ⚖️ Trade-offs to remember

**Pros**
- App stays small and language-agnostic.
- Policies (TLS, retries, logging) are consistent across the fleet.
- Sidecars are upgraded independently of the app.

**Cons**
- Every pod pays a CPU + memory cost (×N pods). In huge clusters this adds up — it's why some teams move toward **ambient mesh** (Istio ambient, Cilium) where the mesh runs per-node instead of per-pod.
- Extra localhost hop adds latency (usually sub-millisecond, but real).
- One more moving part to operate, version, and debug (`kubectl logs pod -c istio-proxy`).
- Startup ordering matters: if the app calls out before the sidecar is ready, you get failures. Tools like Istio's `holdApplicationUntilProxyStarts` exist just to fix this.

## 🧠 Rule of thumb
> Put in the sidecar anything that is *about the network, not the domain*: TLS, retries, service discovery, telemetry, log shipping, secret refresh. Keep business logic in the app.